## hust models

In [9]:
import argparse
from model_collection.Resnet import ResNet, BasicBlock
from model_collection.Sincnet import Sincnet,Sinc_net_m
from model_collection.WKN import WKN,WKN_m
from model_collection.EELM import Dong_ELM
from model_collection.MWA_CNN import A_cSE,Huan_net
from model_collection.TFN.Models.TFN import TFN_Morlet
from model_collection.MCN.models import MCN_GFK, MultiChannel_MCN_GFK
from model_collection.MCN.models import MCN_WFK,MultiChannel_MCN_WFK
from trainer.trainer_basic import Basic_plmodel
from einops import rearrange
import torch
from pytorch_lightning import seed_everything
from configs.config import parse_arguments,config_network
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

parser = argparse.ArgumentParser(description='comparison model')
parser.add_argument('--config_dir', type=str, default='configs/a_031_HUST/config_MCN_basic.yaml',help='The directory of the configuration file')
# parser.add_argument('--config_dir', type=str, default='configs/a_031_HUST/config_MWA_CNN_basic.yaml',help='The directory of the configuration file')
# parser.add_argument('--config_dir', type=str, default='configs/a_031_HUST/config_TFN_basic.yaml',help='The directory of the configuration file')

# 适用于jupyter
meta_args = parser.parse_known_args()[0]
config_dir = meta_args.config_dir
configs,args,path,name = parse_arguments(config_dir, 0)
ff = np.arange(0, args.in_dim//2 + 1) / args.in_dim//2 + 1
MODEL_DICT = {
            'Resnet': lambda args: ResNet(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'WKN_m': lambda args: WKN_m(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'Sinc_net_m': lambda args: Sinc_net_m(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'Huan_net': lambda args: Huan_net(input_size=args.in_channels, num_class=args.num_classes),
            'TFN_Morlet': lambda args: TFN_Morlet(in_channels=args.in_channels, out_channels=args.num_classes),
            'MCN_GFK': lambda args: MultiChannel_MCN_GFK(ff=ff, in_channels=args.in_channels, num_MFKs=8, num_classes=args.num_classes),
        }

model_plain = MODEL_DICT[args.model](args)
model = Basic_plmodel(model_plain, args)
state_dict = torch.load("./save/test/model_mcn_hust_20hz.ckpt")
# state_dict = torch.load("./save/test/model_mwacnn_hust_20hz.ckpt")
# state_dict = torch.load("./save/test/model_tfn_hust_20hz.ckpt")
model.load_state_dict(state_dict['state_dict'])
# print(model)


Running experiment: model_MCN_GFKtime28-15-51-26_datasetHUST_031_Basic_it0


C:\Users\CCSLab\AppData\Local\Temp\ipykernel_18328\2665369159.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("./save/test/model_mcn_hust_20hz.c

<All keys matched successfully>

## plot function

In [1]:
def get_mcn_signal(model, signal):
    out = torch.cat([model.features[i](signal[:, :, i].unsqueeze(1)) for i in range(len(model.features))], dim=1)
    out = model.conv(out.real)
    return out

def get_tfn_signal(model, signal):
    x = model.funconv(signal)
    x = model.layer1(x)
    x = model.layer2(x)
    out = model.layer3(x)
    # out = model.layer4(x)
    return out

def get_mwa_cnn_signal(model, signal):
    input = rearrange(signal, 'b l c -> b c l')
    DMT_yl,DMT_yh = model.DWT0(input)
    output = torch.cat([DMT_yl,DMT_yh[0]], dim=1)
        
    output = model.SConv1(output)
    DMT_yl,DMT_yh = model.DWT1(output)
    output = torch.cat([DMT_yl,DMT_yh[0]], dim=1)
    output = model.dropout1(output)
    output = model.cSE1(output)
        
    output = model.SConv2(output)
    DMT_yl,DMT_yh = model.DWT2(output)
    output = torch.cat([DMT_yl,DMT_yh[0]], dim=1) 
    output = model.dropout2(output)
    output = model.cSE2(output)
        
    output = model.SConv3(output)
    DMT_yl,DMT_yh = model.DWT3(output)
    output = torch.cat([DMT_yl,DMT_yh[0]], dim=1) 
    output = model.dropout3(output)
    output = model.cSE3(output)
        
    output = model.SConv6(output)
    return output

def compute_frequency_domain(signal):
    fft_values = torch.fft.rfft(signal)
    fft_values = fft_values.detach().cpu().numpy()
    # 去除直流项
    fft_values[0] = 0
    return np.abs(fft_values)

def plot_signal(signal, savename):
    plt.rcParams['font.family'] = 'Times New Roman'
    
    fig, axs = plt.subplots(4, 2, figsize=(16, 4))
   
    for i in range(4):
        signal_time = F.normalize(signal[i,:],dim=0)
        t = np.linspace(0, 1, signal_time.shape[0])
        signal_time = signal_time.detach().cpu().numpy()
        signal_fft = compute_frequency_domain(signal[i,:])
        axs[i,0].plot(t, signal_time,linewidth=1.0 ,c='peru')
        axs[i,1].plot(signal_fft,linewidth=1.0 ,c='darkolivegreen')
        if i == 3:
            axs[i,0].set_xlabel('Time (s)')
            axs[i,1].set_xlabel('Frequency (Hz)')
        else:
            axs[i,0].set_xticks([])
            axs[i,1].set_xticks([])
    fig.subplots_adjust(wspace=0.1, hspace=0.1)
    # fig.tight_layout()
    plt.savefig(savename)

def plot_frequency_MCN(signal, savename):
    plt.rcParams['font.family'] = 'Times New Roman'
    
    fig, axs = plt.subplots(2, 2, figsize=(16, 4))
   
    for i in range(4):
        signal_fft = compute_frequency_domain(signal[i,:])
        axs[i//2,i%2].plot(signal_fft,linewidth=1.0 ,c='darkolivegreen')
        if i in [0,1]:
            axs[i//2,i%2].set_xticks([])
        else:
            axs[i//2,i%2].set_xlabel('Frequency (Hz)')
            # axs[i//2,i%2].set_ylabel('Amplitude')
    fig.tight_layout()
    plt.savefig(savename)

## hust data and plot

In [ ]:
test_data = np.load("C:/Users/CCSLab/Desktop/HUST_bearing/HUST_bearing_20Hz_data.npy")
test_signal = torch.from_numpy(test_data).cuda().float()
fault_type_list = ['Norm','Inner Fault','Inner Fault','Outer Fault','Outer Fault','Ball Fault','Ball Fault','Combination Fault','Combination Fault']
if args.model == 'MCN_GFK':
    output_signal = get_mcn_signal(model.network.cuda(), test_signal)[100,:,:]
    print(output_signal.shape)
    plot_frequency_MCN(output_signal, 'save/figure/mcn/signal.svg')
    
if args.model == 'TFN_Morlet':
    # print(model.network)
    output_signal = get_tfn_signal(model.network.cuda(), test_signal)[100,:,:]
    print(output_signal.shape)
    plot_signal(output_signal, 'save/figure/tfn/signal.svg')

if args.model == 'Huan_net':
    # print(model.network)
    output_signal = get_mwa_cnn_signal(model.network.cuda(), test_signal)[100,:,:]
    print(output_signal.shape)
    plot_signal(output_signal, 'save/figure/mwacnn/signal.svg')


## seu models

In [ ]:
import argparse
from model_collection.Resnet import ResNet, BasicBlock
from model_collection.Sincnet import Sincnet,Sinc_net_m
from model_collection.WKN import WKN,WKN_m
from model_collection.EELM import Dong_ELM
from model_collection.MWA_CNN import A_cSE,Huan_net
from model_collection.TFN.Models.TFN import TFN_Morlet
from model_collection.MCN.models import MCN_GFK, MultiChannel_MCN_GFK
from model_collection.MCN.models import MCN_WFK,MultiChannel_MCN_WFK
from trainer.trainer_basic import Basic_plmodel
from einops import rearrange
import torch
from pytorch_lightning import seed_everything
from configs.config import parse_arguments,config_network
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

parser = argparse.ArgumentParser(description='comparison model')
parser.add_argument('--config_dir', type=str, default='configs/a_010_SEU/config_MCN_basic.yaml',help='The directory of the configuration file')
# parser.add_argument('--config_dir', type=str, default='configs/a_010_SEU/config_MWA_CNN_basic.yaml',help='The directory of the configuration file')
# parser.add_argument('--config_dir', type=str, default='configs/a_010_SEU/config_TFN_basic.yaml',help='The directory of the configuration file')

# 适用于jupyter
meta_args = parser.parse_known_args()[0]
config_dir = meta_args.config_dir
configs,args,path,name = parse_arguments(config_dir, 0)
ff = np.arange(0, args.in_dim//2 + 1) / args.in_dim//2 + 1
MODEL_DICT = {
            'Resnet': lambda args: ResNet(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'WKN_m': lambda args: WKN_m(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'Sinc_net_m': lambda args: Sinc_net_m(BasicBlock, [2, 2, 2, 2], in_channel=args.in_channels, num_class=args.num_classes),
            'Huan_net': lambda args: Huan_net(input_size=args.in_channels, num_class=args.num_classes),
            'TFN_Morlet': lambda args: TFN_Morlet(in_channels=args.in_channels, out_channels=args.num_classes),
            'MCN_GFK': lambda args: MultiChannel_MCN_GFK(ff=ff, in_channels=args.in_channels, num_MFKs=8, num_classes=args.num_classes),
        }

model_plain = MODEL_DICT[args.model](args)
model = Basic_plmodel(model_plain, args)
state_dict = torch.load("./save/test/model_mcn_hust_20hz.ckpt")
# state_dict = torch.load("./save/test/model_mwacnn_hust_20hz.ckpt")
model.load_state_dict(state_dict['state_dict'])